In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Permutation-Based Falsification Test for Causal Graphs\n",
    "\n",
    "This notebook demonstrates how to use pgmpy's permutation-based falsification test to evaluate whether causal graphs are consistent with observational data.\n",
    "\n",
    "## Background\n",
    "\n",
    "The permutation-based falsification test is a statistical method that:\n",
    "1. **Tests Falsifiability**: Determines if a graph is informative enough to be tested\n",
    "2. **Tests Falsification**: Evaluates if the graph performs significantly better than random permutations\n",
    "\n",
    "**Reference**: Eulig, E., et al. \"Toward falsifying causal graphs using a permutation-based test\" AAAI 2025.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "from pgmpy.models import DiscreteBayesianNetwork\n",
    "from pgmpy.metrics import permutation_based_falsification_test\n",
    "from pgmpy.utils import get_example_model\n",
    "\n",
    "# Set random seed for reproducibility\n",
    "np.random.seed(42)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Example 1: Testing a Correct Model\n",
    "\n",
    "Let's start with a simple causal chain and test whether it's consistent with data generated from the same structure."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create a simple causal chain: Education -> Income -> Health\n",
    "correct_model = DiscreteBayesianNetwork([\n",
    "    ('Education', 'Income'),\n",
    "    ('Income', 'Health')\n",
    "])\n",
    "\n",
    "print(\"Model structure:\", correct_model.edges())\n",
    "print(\"Nodes:\", correct_model.nodes())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate synthetic data that follows this causal model\n",
    "n_samples = 1000\n",
    "\n",
    "# Education is independent (0 = low, 1 = high)\n",
    "education = np.random.binomial(1, 0.4, n_samples)\n",
    "\n",
    "# Income depends on Education\n",
    "income = np.random.binomial(1, 0.2 + 0.5 * education, n_samples)\n",
    "\n",
    "# Health depends on Income\n",
    "health = np.random.binomial(1, 0.3 + 0.4 * income, n_samples)\n",
    "\n",
    "correct_data = pd.DataFrame({\n",
    "    'Education': education,\n",
    "    'Income': income,\n",
    "    'Health': health\n",
    "})\n",
    "\n",
    "print(\"Data shape:\", correct_data.shape)\n",
    "print(\"\\nData summary:\")\n",
    "print(correct_data.describe())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Test the correct model against its own data\n",
    "result_correct = permutation_based_falsification_test(\n",
    "    correct_model, \n",
    "    correct_data,\n",
    "    n_permutations=100,\n",
    "    return_summary=True\n",
    ")\n",
    "\n",
    "print(\"=== Results for Correct Model ===\")\n",
    "print(f\"Falsifiable: {result_correct['falsifiable']} (p = {result_correct['p_value_falsifiable']:.3f})\")\n",
    "print(f\"Falsified: {result_correct['falsified']} (p = {result_correct['p_value_falsified']:.3f})\")\n",
    "print(f\"LMC violations: {result_correct['lmc_violations']}\")\n",
    "print(f\"Same MEC count: {result_correct['same_mec_count']}/{result_correct['n_permutations']}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Example 2: Testing an Incorrect Model\n",
    "\n",
    "Now let's test a model that contradicts the data generation process."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create an incorrect model - reverse the causal direction\n",
    "wrong_model = DiscreteBayesianNetwork([\n",
    "    ('Health', 'Income'),\n",
    "    ('Income', 'Education')\n",
    "])\n",
    "\n",
    "print(\"Wrong model structure:\", wrong_model.edges())\n",
    "\n",
    "# Test the wrong model against the same data\n",
    "result_wrong = permutation_based_falsification_test(\n",
    "    wrong_model, \n",
    "    correct_data,\n",
    "    n_permutations=100,\n",
    "    return_summary=True\n",
    ")\n",
    "\n",
    "print(\"\\n=== Results for Wrong Model ===\")\n",
    "print(f\"Falsifiable: {result_wrong['falsifiable']} (p = {result_wrong['p_value_falsifiable']:.3f})\")\n",
    "print(f\"Falsified: {result_wrong['falsified']} (p = {result_wrong['p_value_falsified']:.3f})\")\n",
    "print(f\"LMC violations: {result_wrong['lmc_violations']}\")\n",
    "print(f\"Same MEC count: {result_wrong['same_mec_count']}/{result_wrong['n_permutations']}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Example 3: Visualizing the Results"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Compare violation distributions\n",
    "fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))\n",
    "\n",
    "# Plot for correct model\n",
    "correct_violations = result_correct['summary']['permutation_violations']\n",
    "ax1.hist(correct_violations, bins=20, alpha=0.7, color='green', edgecolor='black')\n",
    "ax1.axvline(result_correct['lmc_violations'], color='red', linestyle='--', \n",
    "           label=f'Observed: {result_correct[\"lmc_violations\"]}')\n",
    "ax1.set_title('Correct Model\\nViolation Distribution')\n",
    "ax1.set_xlabel('LMC Violations')\n",
    "ax1.set_ylabel('Frequency')\n",
    "ax1.legend()\n",
    "ax1.grid(True, alpha=0.3)\n",
    "\n",
    "# Plot for wrong model\n",
    "wrong_violations = result_wrong['summary']['permutation_violations']\n",
    "ax2.hist(wrong_violations, bins=20, alpha=0.7, color='red', edgecolor='black')\n",
    "ax2.axvline(result_wrong['lmc_violations'], color='blue', linestyle='--',\n",
    "           label=f'Observed: {result_wrong[\"lmc_violations\"]}')\n",
    "ax2.set_title('Wrong Model\\nViolation Distribution')\n",
    "ax2.set_xlabel('LMC Violations')\n",
    "ax2.set_ylabel('Frequency')\n",
    "ax2.legend()\n",
    "ax2.grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Example 4: Testing with Continuous Data\n",
    "\n",
    "The test also supports continuous data using Pearson correlation for conditional independence testing."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate continuous data following the same causal structure\n",
    "n_samples = 800\n",
    "\n",
    "# Education (standardized)\n",
    "education_cont = np.random.normal(0, 1, n_samples)\n",
    "\n",
    "# Income depends on Education + noise\n",
    "income_cont = 0.6 * education_cont + np.random.normal(0, 0.8, n_samples)\n",
    "\n",
    "# Health depends on Income + noise\n",
    "health_cont = 0.5 * income_cont + np.random.normal(0, 0.7, n_samples)\n",
    "\n",
    "continuous_data = pd.DataFrame({\n",
    "    'Education': education_cont,\n",
    "    'Income': income_cont,\n",
    "    'Health': health_cont\n",
    "})\n",
    "\n",
    "print(\"Continuous data summary:\")\n",
    "print(continuous_data.describe())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Test with continuous data using Pearson correlation\n",
    "result_continuous = permutation_based_falsification_test(\n",
    "    correct_model,\n",
    "    continuous_data,\n",
    "    ci_test='pearsonr',\n",
    "    n_permutations=50,\n",
    "    significance_level=0.05\n",
    ")\n",
    "\n",
    "print(\"=== Results for Continuous Data ===\")\n",
    "print(f\"Falsifiable: {result_continuous['falsifiable']} (p = {result_continuous['p_value_falsifiable']:.3f})\")\n",
    "print(f\"Falsified: {result_continuous['falsified']} (p = {result_continuous['p_value_falsified']:.3f})\")\n",
    "print(f\"LMC violations: {result_continuous['lmc_violations']}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Example 5: Testing with Real pgmpy Models"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "try:\n",
    "    # Load a real pgmpy example model\n",
    "    cancer_model = get_example_model('cancer')\n",
    "    print(\"Cancer model nodes:\", cancer_model.nodes())\n",
    "    print(\"Cancer model edges:\", cancer_model.edges())\n",
    "    \n",
    "    # Generate data from the model\n",
    "    cancer_data = cancer_model.simulate(1000)\n",
    "    \n",
    "    # Test the model against its own data\n",
    "    result_cancer = permutation_based_falsification_test(\n",
    "        cancer_model,\n",
    "        cancer_data,\n",
    "        n_permutations=50\n",
    "    )\n",
    "    \n",
    "    print(\"\\n=== Results for Cancer Model ===\")\n",
    "    print(f\"Falsifiable: {result_cancer['falsifiable']} (p = {result_cancer['p_value_falsifiable']:.3f})\")\n",
    "    print(f\"Falsified: {result_cancer['falsified']} (p = {result_cancer['p_value_falsified']:.3f})\")\n",
    "    print(f\"LMC violations: {result_cancer['lmc_violations']}\")\n",
    "    \n",
    "except Exception as e:\n",
    "    print(f\"Could not load example model: {e}\")\n",
    "    print(\"This is normal if example models are not available in your pgmpy installation.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Interpretation Guide\n",
    "\n",
    "### Understanding the Results:\n",
    "\n",
    "1. **Falsifiable = True**: The graph is informative enough to be tested (has non-trivial independence constraints)\n",
    "2. **Falsified = True**: The graph performs significantly worse than random permutations (likely incorrect)\n",
    "3. **p_value_falsifiable**: Probability that random permutations have the same Markov equivalence class\n",
    "4. **p_value_falsified**: Probability that random permutations perform as poorly as the tested graph\n",
    "\n",
    "### Decision Framework:\n",
    "\n",
    "- **Falsifiable = False**: Graph is not testable (too few constraints)\n",
    "- **Falsifiable = True, Falsified = False**: Graph passes the test (consistent with data)\n",
    "- **Falsifiable = True, Falsified = True**: Graph fails the test (likely incorrect)\n",
    "\n",
    "### Best Practices:\n",
    "\n",
    "- Use at least 100 permutations for reliable results\n",
    "- Consider the significance level (default 0.05)\n",
    "- Test multiple model hypotheses\n",
    "- Validate with domain knowledge"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Summary\n",
    "\n",
    "This notebook demonstrated:\n",
    "\n",
    "1. ✅ **Basic Usage**: How to apply the permutation-based falsification test\n",
    "2. ✅ **Model Comparison**: Comparing correct vs. incorrect models\n",
    "3. ✅ **Continuous Data**: Using the test with continuous variables\n",
    "4. ✅ **Real Models**: Testing with pgmpy example models\n",
    "5. ✅ **Interpretation**: Understanding and visualizing results\n",
    "\n",
    "The permutation-based falsification test provides a principled way to evaluate causal graphs by comparing them against meaningful baselines, helping researchers avoid false discoveries in causal structure learning."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.5"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}